# Z-Image-Turbo GGUF Q6_K
## Dual-GPU Image Generation · Kaggle Production Demo

Run the image-generation pipeline **directly in this notebook** on two NVIDIA Tesla T4 GPUs. The demonstration encodes two prompts on GPU 0, launches a separate GGUF Q6_K inference model on **each physical GPU**, renders two new 1024 × 1024 images, and presents the results, prompts, seeds, execution logs, and performance measurements below.

**The notebook is the demonstration.** Z-Image Studio and the published image galleries are optional browser experiences provided afterward; their jobs and images are not used as evidence for the notebook's inference run.

**Project:** [Source repository](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU) · **Model:** GGUF Q6_K · **Resolution:** 1024 × 1024 · **Inference:** 9 steps · **GPUs:** T4 ×2

---

# Z-Image-Turbo GGUF Q6_K
## Sinh ảnh trên hai GPU · Production Demo Kaggle

Notebook này **trực tiếp thực hiện quá trình sinh ảnh** bằng hai GPU NVIDIA Tesla T4. Hệ thống mã hóa hai prompt trên GPU 0, khởi tạo **hai instance model GGUF Q6_K độc lập** trên GPU 0 và GPU 1, sinh hai ảnh mới kích thước 1024 × 1024, sau đó hiển thị ảnh, prompt, seed, log thực thi và hiệu suất ngay bên dưới.

**Notebook là phần trình diễn chính.** Z-Image Studio và hai bộ sưu tập ảnh là các trải nghiệm bổ sung trên trình duyệt. Kết quả từ Studio không được sử dụng thay thế bằng chứng sinh ảnh của notebook.

**Mã nguồn:** [GitHub](https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU) · **Model:** GGUF Q6_K · **Độ phân giải:** 1024 × 1024 · **Số bước:** 9 · **GPU:** T4 ×2


## Before you run · Kaggle setup

1. Select **Settings → Accelerator → GPU T4 ×2** and enable **Internet**.
2. Use **Add Input → Models** to attach [Z-Image-Turbo GGUF](https://www.kaggle.com/models/dangkhoa2016/unsloth-z-image-turbo-gguf/gguf/q6-k), variation **q6-k**, version **1**.
3. Use **Add Input → Datasets** to attach [Z-Image Diffusers Support Components](https://www.kaggle.com/datasets/dangkhoa2016/z-image-turbo-diffusers-support-components) (text encoder, tokenizer, VAE, and scheduler files).
4. Run the notebook cells in order. The source code is retrieved directly from GitHub, **not** from an application-source dataset.
5. A first successful run renders a pair automatically. Re-running the notebook in the same `/kaggle/working` directory **reuses its verified output** rather than spending GPU time twice.

**Where Kaggle mounts the inputs:**

| Resource | Kaggle location |
|---|---|
| GGUF model | `/kaggle/input/models/dangkhoa2016/unsloth-z-image-turbo-gguf/gguf/q6-k/1/z-image-turbo-Q6_K.gguf` |
| Support components | `/kaggle/input/datasets/dangkhoa2016/z-image-turbo-diffusers-support-components/z-image-turbo-diffusers-support` |

---

## Chuẩn bị · Thiết lập Kaggle

1. Vào **Settings → Accelerator → GPU T4 ×2** và bật **Internet**.
2. Chọn **Add Input → Models** và thêm [Z-Image-Turbo GGUF](https://www.kaggle.com/models/dangkhoa2016/unsloth-z-image-turbo-gguf/gguf/q6-k), biến thể **q6-k**, phiên bản **1**.
3. Chọn **Add Input → Datasets** và thêm [Z-Image Diffusers Support Components](https://www.kaggle.com/datasets/dangkhoa2016/z-image-turbo-diffusers-support-components), chứa text encoder, tokenizer, VAE và scheduler.
4. Chạy tuần tự các cell. Mã nguồn ứng dụng được lấy trực tiếp từ GitHub, **không** cần dataset riêng cho source code.
5. Lần chạy đầu tiên sẽ tự sinh cặp ảnh. Chạy lại trong cùng thư mục `/kaggle/working` sẽ **sử dụng kết quả đã xác minh**, không tự tốn GPU sinh lại.

Hai đường dẫn Kaggle Inputs đã được liệt kê ở bảng phía trên.


## 01 · Load the published application source

Check out the reviewed source revision from GitHub. The notebook verifies the commit and preserves existing files without resetting your Kaggle workspace.

---

## 01 · Tải mã nguồn ứng dụng

Lấy phiên bản source đã được xác định trên GitHub. Notebook kiểm tra commit và giữ nguyên dữ liệu có sẵn, không tự reset thư mục làm việc trên Kaggle.


In [ ]:
from pathlib import Path
import importlib.util, os, socket, subprocess, sys, uuid, urllib.request

REPO_URL = "https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU.git"
SOURCE_REF = "fix/studio-gpu-startup-timeout"
SOURCE_SHA = "185d9950081c0ebc9421a3b14ed84127dbff77ea"  # Verified GPU code plus bounded Studio startup fix
BASE_PROJECT = Path("/kaggle/working/zimage-source-demo")
PROJECT = BASE_PROJECT
REUSE_LIVE_STUDIO = False
STUDIO_PORTS = {"api": 18788, "gateway": 18789}

def git(project, *args):
    return subprocess.run(
        ["git", "-C", str(project), *args], check=True,
        capture_output=True, text=True, timeout=30
    ).stdout.strip()

def verified_checkout(project):
    """Verify public source identity without resetting, fetching or modifying it."""
    project = Path(project).resolve()
    if not (project / ".git").is_dir():
        return False, "Not a Git source checkout", None
    try:
        remote = git(project, "config", "--get", "remote.origin.url")
        normal = lambda x: x.removesuffix(".git").rstrip("/")
        if normal(remote) != normal(REPO_URL):
            return False, "Git remote does not match this project", None
        revision = git(project, "rev-parse", "HEAD")
        target = git(project, "rev-parse", (SOURCE_SHA or SOURCE_REF) + "^{commit}")
        if revision != target:
            return False, "Source revision differs from published demo revision", None
        for args in (("diff", "--quiet"), ("diff", "--cached", "--quiet")):
            if subprocess.run(["git", "-C", str(project), *args],
                              capture_output=True, timeout=15).returncode != 0:
                return False, "Tracked source has local edits", None
        return True, "SOURCE_SHA=" + revision, {"sha": revision, "remote": remote}
    except (OSError, subprocess.SubprocessError) as exc:
        return False, "Cannot verify Git checkout: " + type(exc).__name__, None

def active_ports():
    found = []
    for port in STUDIO_PORTS.values():
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(0.5)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                found.append(port)
    return found

def listener_pids(port):
    """Resolve socket ownership from procfs rather than potentially stale PID files."""
    inodes = set()
    for name in ("/proc/net/tcp", "/proc/net/tcp6"):
        try:
            lines = Path(name).read_text().splitlines()[1:]
        except OSError:
            continue
        for line in lines:
            fields = line.split()
            if len(fields) > 9 and fields[3] == "0A":
                try:
                    if int(fields[1].rsplit(":", 1)[-1], 16) == port:
                        inodes.add("socket:[" + fields[9] + "]")
                except ValueError:
                    continue
    owners = set()
    if not inodes:
        return owners
    for proc in Path("/proc").iterdir():
        if not proc.name.isdigit():
            continue
        try:
            for fd in (proc / "fd").iterdir():
                try:
                    target = os.readlink(fd)
                except OSError:
                    continue
                if target in inodes:
                    owners.add(int(proc.name))
                    break
        except OSError:
            continue
    return owners

def inspect_service(service, port):
    """Return observed running source path only when process identity is verified."""
    module = "demo_api:app" if service == "api" else "gateway:app"
    suffix = Path("production-system/studio-demo") if service == "api" else Path("product-studio")
    notes = []
    pids = sorted(listener_pids(port))
    if not pids:
        return None, service + ": no accessible listening-process owner on port " + str(port)
    for pid in pids:
        proc = Path("/proc") / str(pid)
        try:
            argv = [x.decode(errors="replace") for x in
                    (proc / "cmdline").read_bytes().split(b"\x00") if x]
            cwd = (proc / "cwd").resolve(strict=True)
            state = (proc / "stat").read_text().rsplit(")", 1)[1].strip().split()[0]
            environment = (proc / "environ").read_bytes().split(b"\x00")
        except OSError as exc:
            notes.append("PID " + str(pid) + ": cannot inspect procfs (" + type(exc).__name__ + ")")
            continue
        if state in ("Z", "X"):
            notes.append("PID " + str(pid) + ": not running")
            continue
        if not (module in argv and "uvicorn" in argv and "--port" in argv and
                argv.index("--port") + 1 < len(argv) and
                argv[argv.index("--port") + 1] == str(port)):
            notes.append("PID " + str(pid) + ": command/module is not the expected Studio service")
            continue
        if cwd.name != suffix.name or not str(cwd).endswith("/" + str(suffix)):
            notes.append("PID " + str(pid) + ": working directory is not Studio source: " + str(cwd))
            continue
        project = cwd.parents[1] if service == "api" else cwd.parent
        run_db = project / "production-system/b3-studio-demo-live-20261016/api.sqlite3"
        key = "ZIMAGE_B1_DB" if service == "api" else "ZIMAGE_STUDIO_DB"
        expected = (key + "=" + str(run_db)).encode()
        if expected not in environment:
            notes.append("PID " + str(pid) + ": " + key + " does not match its source checkout")
            continue
        if not run_db.is_file():
            notes.append("PID " + str(pid) + ": Studio database missing")
            continue
        return project.resolve(), "PID " + str(pid) + " port " + str(port)
    return None, "; ".join(notes)

def locate_live_studio():
    """Find the real source folder of an existing Studio, including clean-* clones."""
    api_project, api_info = inspect_service("api", STUDIO_PORTS["api"])
    gateway_project, gateway_info = inspect_service("gateway", STUDIO_PORTS["gateway"])
    if api_project is None or gateway_project is None:
        return None, "API: " + api_info + " | GATEWAY: " + gateway_info
    if api_project != gateway_project:
        return None, "API and gateway were started from DIFFERENT source checkouts: " + str(api_project) + " / " + str(gateway_project)
    run = api_project / "production-system/b3-studio-demo-live-20261016"
    if not (run / "local-secrets.json").is_file():
        return None, "Existing Studio login configuration not found"
    for name, endpoint in (("api", "/health"), ("gateway", "/auth/session")):
        try:
            with urllib.request.urlopen(
                "http://127.0.0.1:" + str(STUDIO_PORTS[name]) + endpoint,
                timeout=5
            ) as response:
                if response.status != 200:
                    return None, name + " HTTP returned " + str(response.status)
        except Exception as exc:
            return None, name + " HTTP check failed: " + type(exc).__name__
    return api_project, "API=" + api_info + " GATEWAY=" + gateway_info

ports_in_use = active_ports()
if ports_in_use:
    if set(ports_in_use) != set(STUDIO_PORTS.values()):
        raise RuntimeError(
            "PARTIAL_STUDIO_SESSION: only ports " + str(ports_in_use) + " are reachable. "
            "The existing services were not modified. For a clean start, use the "
            "Kaggle UI to restart the session after saving your files."
        )
    live_project, discovery = locate_live_studio()
    if live_project is None:
        raise RuntimeError(
            "STUDIO_REUSE_NOT_VERIFIED: " + discovery + ". "
            "No running process or local files were changed. Save the required "
            "Kaggle outputs before choosing Restart Session for a clean run."
        )
    ok, reason, info = verified_checkout(live_project)
    if not ok:
        raise RuntimeError(
            "STUDIO_SOURCE_NOT_VERIFIED: " + reason + "; running checkout: "
            + str(live_project) + ". Existing services were not changed."
        )
    PROJECT = live_project
    REUSE_LIVE_STUDIO = True
    print("EXISTING_STUDIO_REUSED", discovery, flush=True)
else:
    ok, reason, info = verified_checkout(BASE_PROJECT)
    if ok:
        PROJECT = BASE_PROJECT
        print("EXISTING_SOURCE_REUSED", reason, flush=True)
    else:
        if BASE_PROJECT.exists():
            PROJECT = Path("/kaggle/working") / ("zimage-source-demo-clean-" + uuid.uuid4().hex[:8])
            print("PREVIOUS_SOURCE_PRESERVED", BASE_PROJECT, flush=True)
        assert not PROJECT.exists(), "Refusing to overwrite an existing path"
        subprocess.run(["git", "clone", "--branch", SOURCE_REF, REPO_URL, str(PROJECT)],
                       check=True, timeout=180)
        if SOURCE_SHA:
            subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", SOURCE_SHA],
                           check=True, timeout=30)
        ok, reason, info = verified_checkout(PROJECT)
        if not ok:
            raise RuntimeError("FRESH_SOURCE_VERIFICATION_FAILED: " + reason)
        print("SOURCE_CLONED_FRESH", PROJECT, flush=True)

assert (PROJECT / "product-studio/index.html").is_file(), "Application source incomplete"
assert info and info["sha"], "Source revision verification failed"
print("SOURCE_READY", PROJECT, flush=True)
print("SOURCE_SHA", info["sha"], flush=True)
print("STUDIO_SESSION_MODE", "REUSE_VERIFIED" if REUSE_LIVE_STUDIO else "FRESH_OR_RESUME", flush=True)


## 02 · Verify the GPUs and attached inputs

Confirm both Tesla T4 devices, the selected GGUF Q6_K weights, and the support components before initializing the image-generation pipeline.

---

## 02 · Kiểm tra GPU và tài nguyên đầu vào

Xác minh hai GPU Tesla T4, trọng số model GGUF Q6_K và các thành phần hỗ trợ trước khi bắt đầu quá trình sinh ảnh.


In [ ]:
import subprocess,sys,importlib.metadata as md
model=Path("/kaggle/input/models/dangkhoa2016/unsloth-z-image-turbo-gguf/gguf/q6-k/1/z-image-turbo-Q6_K.gguf")
support=Path("/kaggle/input/datasets/dangkhoa2016/z-image-turbo-diffusers-support-components/z-image-turbo-diffusers-support")
assert model.is_file(),f"Attach Kaggle model first: {model}"
assert all((support/component).is_dir() for component in ("text_encoder", "tokenizer", "vae")), (
    "Missing Kaggle support dataset or required text_encoder/tokenizer/vae directory: " + str(support)
)
gpus=subprocess.check_output(["nvidia-smi","-L"],text=True)
assert gpus.count("Tesla T4")>=2,"Switch Kaggle accelerator to T4 ×2"
print("GPU_T4X2_READY",gpus)
print("MODEL_INPUT_ATTACHED:", model)
print("MODEL_VARIATION_VERSION:", "GGUF / q6-k / 1")
print("SUPPORT_DATASET_ATTACHED:", support)
print("SUPPORT_COMPONENTS:", "text_encoder, tokenizer, vae")
print("GGUF_BYTES",model.stat().st_size)
for key in ("torch","transformers","diffusers","huggingface-hub","torchao","gguf","safetensors","fastapi","uvicorn"):
    try: print(key,md.version(key))
    except md.PackageNotFoundError:print(key,"MISSING")
print("NOTE: Tensor library versions must match the validated runtime. Install missing packages in the next cell; do not randomly upgrade torch.")


## 03 · Prepare the inference runtime

Install and verify the project's SHA256-locked runtime packages. Keep model inference isolated to worker subprocesses rather than loading both model instances into the notebook kernel.

---

## 03 · Chuẩn bị môi trường inference

Cài đặt và xác minh các thư viện đã được khóa SHA256. Hai instance model chạy trong tiến trình riêng, không tải trực tiếp hai model vào kernel notebook.


In [ ]:
# Reconstruct audited upstream vendor packages using exact wheel and file-tree SHA256.
# Source remains GitHub; model and support weights remain separately attached inputs.
import os, subprocess, sys, importlib.util
subprocess.run([sys.executable,str(PROJECT/"scripts"/"bootstrap_vendor.py")],check=True)
VENDOR_PATHS=[f"production-system/{name}" for name in
 ("vendor-shim","vendor-hub128","vendor-diffusers041","vendor-torchao015","vendor-gguf")]
assert all((PROJECT / path).is_dir() for path in VENDOR_PATHS)
# Project-local pinned vendor imports take precedence in subprocesses and future kernels.
VENDORS=[str(PROJECT/path) for path in VENDOR_PATHS]
os.environ["PYTHONPATH"]=os.pathsep.join(VENDORS + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else []))
subprocess.run([sys.executable,str(PROJECT/"scripts"/"verify_vendor_runtime.py")],check=True,env=os.environ.copy())
required=("fastapi","uvicorn","safetensors","transformers","accelerate","httpx","torch")
missing=[package for package in required if importlib.util.find_spec(package) is None]
assert not missing, f"Required runtime packages not installed: {missing}. Provision compatible pinned dependencies before GPU run."
gpu_probe = subprocess.run([sys.executable,'-c',"import torch; assert torch.cuda.device_count()==2; print([torch.cuda.get_device_name(i) for i in range(2)])"],capture_output=True,text=True,timeout=35)
assert gpu_probe.returncode == 0, gpu_probe.stderr
print('TWO_GPUS',gpu_probe.stdout.strip())
os.environ["HF_HUB_OFFLINE"]="1"
os.environ["TRANSFORMERS_OFFLINE"]="1"
print("PREFLIGHT_READY", gpu_probe.stdout.strip())

subprocess.run([sys.executable,str(PROJECT/'scripts/validate_gpu_runtime_imports.py')],check=True,env=os.environ.copy(),timeout=150)
print('GPU_RUNTIME_IMPORT_SMOKE_PASS',flush=True)


## 04 · Encode two prompts on GPU 0

The demonstration first creates verified prompt embeddings on **GPU 0** using the source project's text encoder. This is a separate phase: it exits and releases GPU memory **before** two GGUF inference workers start. The cell prints elapsed time while encoding and stores an integrity-checked inference manifest.

If a previous direct run already produced verified images, the notebook will reuse those results instead of repeating this expensive step. **To accept Studio only, after notebook-native evidence has already been approved, skip Steps 04–06**; running them on a new workspace without a verified report would generate two fresh images.

Existing Studio jobs are managed by Studio, not by this notebook. Old queue records are preserved and do not automatically block the independent demonstration. A live GPU consumer or worker does block it to prevent conflicting GPU workloads.

---

## 04 · Mã hóa hai prompt trên GPU 0

Notebook trước tiên sử dụng text encoder của dự án để mã hóa hai prompt trên **GPU 0**. Đây là giai đoạn độc lập: encoder kết thúc và giải phóng bộ nhớ GPU **trước khi** hai worker GGUF khởi động. Cell hiển thị thời gian đã trôi qua và lưu manifest đã kiểm tra tính toàn vẹn.

Nếu lần chạy trực tiếp trước đó đã có ảnh hợp lệ, notebook sử dụng lại kết quả để tránh lặp lại bước tốn GPU này. **Để nghiệm thu riêng Studio sau khi hai ảnh notebook-native đã được chấp nhận, hãy bỏ qua các bước 04–06**; trên workspace mới không có report hợp lệ, các bước đó sẽ sinh hai ảnh mới.

Các job tồn đọng của Studio được giữ nguyên và không cản trở lượt chạy độc lập của notebook. Tuy nhiên, nếu GPU consumer hoặc worker của Studio còn hoạt động, notebook sẽ dừng để tránh tranh chấp tài nguyên GPU.


In [ ]:
import os, sys, time, json, socket, signal, hashlib, sqlite3, subprocess
from pathlib import Path
from datetime import datetime, timezone

DIRECT_ROOT = Path('/kaggle/working/zimage-notebook-dual-gpu-demo')
DIRECT_REPORT = DIRECT_ROOT / 'report.json'
DIRECT_PROMPTS = [
    'A cinematic photograph of a vintage red sports car parked beside a Mediterranean coastal road at sunrise, detailed paint reflections, realistic natural light, professional automotive photography',
    'A tranquil Japanese mountain village in autumn with a stone bridge over a clear river, colorful maple trees, distant misty mountains, photorealistic environmental detail',
]
DIRECT_SEEDS = (121041, 121042)
PROD = PROJECT / 'production-system'
assert (PROD / 'phase_controller.py').is_file()
assert (PROD / 'dynamic_worker.py').is_file()
assert (PROJECT / 'scripts/validate_gpu_runtime_imports.py').is_file(), 'Choose a source commit with GPU runtime validation available'
VENDOR_ENV = os.environ.copy()
VENDOR_ENV['PYTHONUNBUFFERED'] = '1'
VENDOR_ENV['HF_HUB_OFFLINE'] = '1'
VENDOR_ENV['TRANSFORMERS_OFFLINE'] = '1'

# Existing public images are accepted only after verifying both 1024px PNGs and SHA256.
def direct_previous_result():
    if not DIRECT_REPORT.is_file():
        return None
    report = json.loads(DIRECT_REPORT.read_text(encoding='utf-8'))
    from PIL import Image as PILImage
    if report['source_sha'] != info['sha']:
        # Preserve accepted GPU evidence when only the Studio timeout changed.
        # Never accept a different source revision without verifying the complete diff.
        assert report['source_sha'] == 'ba52ec0be7c1a0396a21fe9fd07afa1c8f022577', (
            'Existing images belong to an unrecognized source revision')
        changed = subprocess.run(
            ['git', '-C', str(PROJECT), 'diff', '--name-only',
             report['source_sha'], info['sha']],
            check=True, capture_output=True, text=True, timeout=30
        ).stdout.splitlines()
        assert changed == ['production-system/studio-demo/demo_consumer.py'], (
            'Cannot reuse GPU proof: inference source changes were detected: ' + repr(changed))
        print('NOTEBOOK_GPU_PROOF_REUSED_AFTER_STUDIO_ONLY_PATCH',
              report['source_sha'], '->', info['sha'], flush=True)
    assert set(int(i['gpu']) for i in report['images']) == {0, 1}
    for image in report['images']:
        p = Path(image['file'])
        assert p.is_file() and p.resolve().is_relative_to(DIRECT_ROOT.resolve())
        assert hashlib.sha256(p.read_bytes()).hexdigest() == image['sha256']
        with PILImage.open(p) as im:
            assert im.format == 'PNG' and im.size == (1024, 1024)
            im.verify()
    return report

DIRECT_CACHED_REPORT = direct_previous_result()
DIRECT_SHOULD_RENDER = DIRECT_CACHED_REPORT is None

if not DIRECT_SHOULD_RENDER:
    print('EXISTING_NOTEBOOK_GPU_IMAGES_VERIFIED — reusing completed direct render; no GPU jobs submitted.', flush=True)
else:
    if DIRECT_ROOT.exists():
        raise RuntimeError('DIRECT_DEMO_PREVIOUS_ATTEMPT_INCOMPLETE: ' + str(DIRECT_ROOT) +
                           '. This notebook will not automatically replay a potentially incomplete GPU run. ' +
                           'Review the previous logs and choose a new output directory only after confirming workers have stopped.')
    for port in (12340, 12341):
        with socket.socket() as probe:
            probe.settimeout(0.3)
            if probe.connect_ex(('127.0.0.1', port)) == 0:
                raise RuntimeError('GPU_WORKER_PORT_OCCUPIED: ' + str(port))
    # Studio and notebook have separate job stores. Stale Studio queue records
    # are NOT proof of live GPU activity and must not block notebook inference.
    # Conversely, a live Studio consumer/encoder/worker must block this step,
    # even if its queue is momentarily empty, to avoid overlapping GPU usage.
    def other_live_inference_processes():
        scripts = {'demo_consumer.py', 'dynamic_worker.py',
                   'phase_controller.py', 'encode_batch_gpu.py',
                   'shared_gpu_supervisor.py'}
        active = []
        for entry in Path('/proc').iterdir():
            if not entry.name.isdecimal():
                continue
            try:
                argv = [x.decode('utf-8', errors='replace') for x in
                        (entry / 'cmdline').read_bytes().split(b'\\0') if x]
                script_names = {Path(part).name for part in argv}
                matched = sorted(script_names & scripts)
                if not matched:
                    continue
                raw_stat = (entry / 'stat').read_text()
                state = raw_stat[raw_stat.rfind(')')+2:].split()[0]
                if state in ('Z', 'X'):
                    continue
                if matched == ['demo_consumer.py'] and '--enable-gpu' not in argv:
                    continue
                active.append({'pid': int(entry.name), 'scripts': matched})
            except (OSError, IndexError, ValueError):
                continue  # A process may exit while /proc is being inspected.
        return active

    studio_databases = sorted(Path('/kaggle/working').glob(
        'zimage-source-demo*/production-system/b3-studio-demo-live-20261016/api.sqlite3'))
    for db_path in studio_databases:
        with sqlite3.connect('file:' + str(db_path) + '?mode=ro', uri=True,
                             timeout=5) as db:
            counts = dict(db.execute(
                "SELECT state, COUNT(*) FROM jobs GROUP BY state").fetchall())
            sessions = []
            if db.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='b112_sessions'").fetchone():
                sessions = db.execute(
                    "SELECT status, COUNT(*) FROM b112_sessions "
                    "WHERE status != 'ARCHIVED' GROUP BY status").fetchall()
        pending = sum(counts.get(state, 0) for state in ('queued', 'running', 'uncertain'))
        if pending or sessions:
            print('STUDIO_HISTORY_LEFT_UNCHANGED',
                  str(db_path), 'job_states=', counts, 'session_states=', sessions,
                  '— notebook will not consume or modify these jobs.', flush=True)

    live_pipelines = other_live_inference_processes()
    if live_pipelines:
        raise RuntimeError('LIVE_GPU_PIPELINE_CONFLICT: ' + json.dumps(live_pipelines) +
                           '. Studio/other inference processes are still active. '
                           'Do not run notebook generation concurrently.')
    gpu_owners = subprocess.run(
        ['nvidia-smi', '--query-compute-apps=pid,gpu_uuid',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True, timeout=15)
    if gpu_owners.returncode != 0:
        raise RuntimeError('GPU_PROCESS_PROBE_FAILED: ' + gpu_owners.stderr[-600:])
    if gpu_owners.stdout.strip():
        raise RuntimeError('GPU_ALREADY_IN_USE_BY_COMPUTE_PROCESSES: ' +
                           gpu_owners.stdout.strip())
    print('DIRECT_NOTEBOOK_GPU_EXCLUSIVITY_CONFIRMED: no live GPU inference '
          'process or CUDA compute owner was found.', flush=True)
    DIRECT_ROOT.mkdir(parents=True, exist_ok=False)
    (DIRECT_ROOT / 'prompts.json').write_text(json.dumps(DIRECT_PROMPTS, indent=2), encoding='utf-8')
    version = subprocess.check_output([sys.executable, '-c',
        "import importlib.metadata as m; print(m.version('diffusers'))"],
        env=VENDOR_ENV, text=True, timeout=30).strip()
    support_manifest = json.loads((PROJECT / 'manifests/support.json').read_text())
    identity = dict(support_manifest_sha256=support_manifest['manifest_sha256'],
                    diffusers_version=version, max_sequence_length=512,
                    text_encoder_dtype='bfloat16')
    (DIRECT_ROOT / 'encoder-identity.json').write_text(json.dumps(identity, indent=2), encoding='utf-8')
    print('NOTEBOOK_DIRECT_GPU_PHASE', 'ENCODING_TWO_PROMPTS', 'physical_gpu=0', flush=True)
    print('PROMPT_0', DIRECT_PROMPTS[0], '\nPROMPT_1', DIRECT_PROMPTS[1], flush=True)

    encoder_cmd = [sys.executable, '-u', str(PROD / 'phase_controller.py'),
                   '--db', str(DIRECT_ROOT / 'registry.sqlite3'),
                   '--cache-root', str(DIRECT_ROOT / 'cache'),
                   '--identity', str(DIRECT_ROOT / 'encoder-identity.json'),
                   '--prompt-json', str(DIRECT_ROOT / 'prompts.json'),
                   '--mode', 'gpu0', '--max-items', '2']
    encoder_log = DIRECT_ROOT / 'encoder.log'
    started = time.monotonic()
    with encoder_log.open('w') as log:
        encoder = subprocess.Popen(encoder_cmd, cwd=PROD, env=VENDOR_ENV,
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        try:
            while encoder.poll() is None:
                elapsed = time.monotonic() - started
                if elapsed > 1500:
                    raise TimeoutError('GPU0 text encoding exceeded 1500 seconds')
                print(f'ENCODER_GPU_0 elapsed={elapsed:.0f}s (not percent)', flush=True)
                time.sleep(15)
        except BaseException:
            if encoder.poll() is None:
                os.killpg(encoder.pid, signal.SIGTERM)
                try: encoder.wait(timeout=15)
                except subprocess.TimeoutExpired: os.killpg(encoder.pid, signal.SIGKILL)
            raise
    DIRECT_ENCODE_SECONDS = round(time.monotonic()-started, 2)
    if encoder.returncode != 0:
        print('\n'.join(encoder_log.read_text(errors='replace').splitlines()[-45:]))
        raise RuntimeError('GPU0_ENCODER_FAILED: see ' + str(encoder_log))
    phase = json.loads((DIRECT_ROOT / 'b16-phase-status.json').read_text())
    assert phase['status'] == 'READY_FOR_INFERENCE_START' and phase['verified_ready'] == 2, phase
    print('GPU0_ENCODER_PASS', f'{DIRECT_ENCODE_SECONDS}s', phase['status'], flush=True)

    # Export verified embeddings for the independent GPU-0 and GPU-1 workers.
    manifest_builder = """
import json,sys
from pathlib import Path
sys.path.insert(0,sys.argv[1])
from embedding_registry import Registry
from dynamic_manifest import export_manifest
root=Path(sys.argv[2]);identity=json.loads((root/'encoder-identity.json').read_text())
prompts=json.loads((root/'prompts.json').read_text())
reg=Registry(root/'registry.sqlite3',root/'cache',identity)
export_manifest(reg,prompts,root/'inference-manifest.json')
print('NOTEBOOK_EMBEDDING_MANIFEST_VERIFIED',flush=True)
"""
    subprocess.run([sys.executable, '-c', manifest_builder, str(PROD), str(DIRECT_ROOT)],
                   env=VENDOR_ENV, check=True, timeout=120)
    subprocess.run([sys.executable, str(PROD / 'gpu_preflight.py'), '--expect-gpu',
                    '--manifest', str(DIRECT_ROOT / 'inference-manifest.json')],
                   cwd=PROD, env=VENDOR_ENV, check=True, timeout=150)
    (DIRECT_ROOT / 'encoder-timing.json').write_text(json.dumps({'seconds': DIRECT_ENCODE_SECONDS}))
    print('NOTEBOOK_GPU_EMBEDDINGS_READY', flush=True)


## 05 · Run two independent GGUF model instances and generate images

This is the **primary live demonstration**, executed directly by the notebook. Two dedicated Python processes each load their own Q6_K transformer and VAE. The GPU assignments are independent: **GPU 0 → worker 0** and **GPU 1 → worker 1**. After both models confirm readiness, the notebook submits one image-generation request to each worker **concurrently** and records the actual worker-measured execution times.

The cell prints startup milestones such as `GGUF_LOAD_BEGIN`, `GGUF_ON_CUDA`, `PIPELINE_READY` and per-worker progress. If a worker cannot start, the notebook stops with diagnostic logs rather than waiting for an unrelated Studio queue. No Studio job is created.

---

## 05 · Chạy hai instance GGUF độc lập và sinh ảnh

Đây là **phần trình diễn inference chính**, được chính notebook khởi chạy. Hai tiến trình Python độc lập lần lượt nạp transformer Q6_K và VAE, với phân công **GPU 0 → worker 0**, **GPU 1 → worker 1**. Khi cả hai model báo sẵn sàng, notebook gửi đồng thời một yêu cầu sinh ảnh đến mỗi worker và ghi nhận thời gian thực tế.

Cell sẽ in các giai đoạn `GGUF_LOAD_BEGIN`, `GGUF_ON_CUDA`, `PIPELINE_READY` cùng tiến độ của từng worker. Nếu model không thể khởi động, notebook báo lỗi và hiển thị log thay vì chờ queue Studio. Phần này không tạo job trong Studio.


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.request import urlopen
from PIL import Image as PILImage
import shutil, urllib.error

if not DIRECT_SHOULD_RENDER:
    print('NOTEBOOK_DIRECT_GENERATION_ALREADY_COMPLETED — images will be displayed below.', flush=True)
else:
    MANIFEST_PATH = DIRECT_ROOT / 'inference-manifest.json'
    assert MANIFEST_PATH.is_file(), 'Run Step 04 before Step 05'
    ARTIFACT_ROOT = PROJECT / 'output'
    assert not ARTIFACT_ROOT.is_symlink(), 'Unsafe artifact root'
    ARTIFACT_ROOT.mkdir(exist_ok=True)
    worker_logs = [DIRECT_ROOT / f'worker-{i}.log' for i in (0,1)]
    worker_processes = []
    worker_handles = []
    started_models = time.monotonic()

    def worker_stage(logpath):
        if not logpath.exists(): return 'STARTING'
        data = logpath.read_text(errors='replace')
        stages = [line for line in data.splitlines() if line.startswith('ZIMAGE_WORKER_STAGE ')]
        if not stages: return 'IMPORTING'
        try: return json.loads(stages[-1].split(' ', 1)[1]).get('stage','UNKNOWN')
        except ValueError: return 'UNREADABLE_STAGE'

    def worker_ready(i):
        try:
            with urlopen(f'http://127.0.0.1:{12340+i}/ready', timeout=2) as response:
                data = json.load(response)
            return data.get('ready') is True and data.get('worker_id') == i
        except (OSError, ValueError):
            return False

    def process_alive(i):
        return worker_processes[i].poll() is None

    def observed_gpu_memory():
        p = subprocess.run(['nvidia-smi', '--query-compute-apps=pid,gpu_uuid,used_gpu_memory',
                            '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=15)
        return p.stdout.strip() if p.returncode == 0 else 'not available'

    try:
        for i in (0,1):
            env = VENDOR_ENV.copy()
            env['CUDA_VISIBLE_DEVICES'] = str(i)
            handle = worker_logs[i].open('w')
            worker_handles.append(handle)
            process = subprocess.Popen(
                [sys.executable, '-u', str(PROD / 'dynamic_worker.py'),
                 '--worker-id', str(i), '--port', str(12340+i), '--manifest', str(MANIFEST_PATH)],
                cwd=PROD, env=env, stdin=subprocess.DEVNULL,
                stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
            worker_processes.append(process)
            print(f'NOTEBOOK_WORKER_{i}_STARTED physical_gpu={i} pid={process.pid}', flush=True)

        startup_deadline = time.monotonic()+360
        last_report = 0
        while True:
            for i in (0,1):
                if not process_alive(i):
                    raise RuntimeError(f'GPU_{i}_MODEL_STARTUP_FAILED exit={worker_processes[i].returncode} stage={worker_stage(worker_logs[i])}')
            ready = [worker_ready(0),worker_ready(1)]
            now = time.monotonic()
            if all(ready): break
            if now >= startup_deadline:
                raise TimeoutError('DUAL_GPU_MODEL_STARTUP_TIMEOUT: ' + str([(i,worker_stage(worker_logs[i])) for i in (0,1)]))
            if now-last_report >= 15:
                print('MODEL_STARTUP', f'elapsed={now-started_models:.0f}s',
                      'GPU0='+worker_stage(worker_logs[0]), 'GPU1='+worker_stage(worker_logs[1]), flush=True)
                last_report = now
            time.sleep(3)
        DIRECT_STARTUP_SECONDS = round(time.monotonic()-started_models,2)
        print('BOTH_MODEL_INSTANCES_READY',
              {'GPU0_PID':worker_processes[0].pid,'GPU1_PID':worker_processes[1].pid,
               'startup_seconds':DIRECT_STARTUP_SECONDS},flush=True)
        print('NVIDIA_SMI_ACTIVE_WORKERS', observed_gpu_memory(),flush=True)
        for i in (0,1):
            print('=== NOTEBOOK GPU',i,'MODEL INITIALIZATION EVIDENCE ===',flush=True)
            stages=[line for line in worker_logs[i].read_text(errors='replace').splitlines()
                    if line.startswith('ZIMAGE_WORKER_STAGE ')]
            assert any('PIPELINE_READY' in line for line in stages), 'Model pipeline is not ready'
            for line in stages:
                print(line,flush=True)

        # Reuse the repository's verification adapter; requests are issued simultaneously.
        def generate_one(i):
            script = r"""
import json,sys
from pathlib import Path
sys.path.insert(0,sys.argv[1])
from worker_adapter import WorkerAdapter
root=Path(sys.argv[2]);project=Path(sys.argv[3]);worker=int(sys.argv[4]);seed=int(sys.argv[5])
manifest=json.loads((root/'inference-manifest.json').read_text())
prompts=json.loads((root/'prompts.json').read_text())
identity=json.loads((root/'encoder-identity.json').read_text())
from embedding_registry import Registry
reg=Registry(root/'registry.sqlite3',root/'cache',identity)
key=reg.key(prompts[worker]);assert key in manifest['entries']
a=WorkerAdapter(worker,f'http://127.0.0.1:{12340+worker}',project/'output',timeout=920)
print(json.dumps(a.generate(key,seed),sort_keys=True))
"""
            cmd = [sys.executable,'-c',script,str(PROD),str(DIRECT_ROOT),str(PROJECT),str(i),str(DIRECT_SEEDS[i])]
            # Separate client processes keep driver/CUDA allocations out of the notebook kernel.
            cp = subprocess.run(cmd,env=VENDOR_ENV,capture_output=True,text=True,timeout=950)
            if cp.returncode:
                raise RuntimeError('GPU_'+str(i)+'_INFERENCE_FAILED: '+cp.stderr[-1200:])
            return json.loads(cp.stdout.strip().splitlines()[-1])

        print('NOTEBOOK_INFERENCE_STARTED gpu0_and_gpu1_concurrent=true', flush=True)
        inference_start = time.monotonic()
        executor = ThreadPoolExecutor(max_workers=2)
        futures = [executor.submit(generate_one,i) for i in (0,1)]
        try:
            next_report=0
            while not all(f.done() for f in futures):
                if any(not process_alive(i) for i in (0,1)):
                    raise RuntimeError('MODEL_WORKER_EXITED_DURING_RENDER')
                now=time.monotonic()
                if now-inference_start > 975:
                    raise TimeoutError('DUAL_GPU_INFERENCE_EXCEEDED_975_SECONDS')
                if now-next_report >= 20:
                    print('NOTEBOOK_RENDER_PROGRESS',f'elapsed={now-inference_start:.0f}s',
                          'completed='+str(sum(f.done() for f in futures))+'/2',flush=True)
                    next_report=now
                time.sleep(3)
            generated = [f.result() for f in futures]
        finally:
            executor.shutdown(wait=False,cancel_futures=True)
        DIRECT_INFERENCE_WALL_SECONDS = round(time.monotonic()-inference_start,2)
        artifacts = []
        for i, entry in enumerate(generated):
            assert entry['worker_id'] == i and entry['seed'] == DIRECT_SEEDS[i]
            assert 'PIPELINE_READY' in worker_logs[i].read_text(errors='replace')
            origin = Path(entry['output_path'])
            destination = DIRECT_ROOT / f'zimage-notebook-gpu-{i}-1024.png'
            with PILImage.open(origin) as im:
                assert im.format=='PNG' and im.size==(1024,1024)
                im.verify()
            shutil.copyfile(origin,destination)
            digest = hashlib.sha256(destination.read_bytes()).hexdigest()
            assert digest == entry['output_sha256']
            artifacts.append({'gpu':i,'pid':worker_processes[i].pid,'seed':DIRECT_SEEDS[i],
                              'prompt':DIRECT_PROMPTS[i], 'file':str(destination), 'sha256':digest,
                              'generation_seconds':entry['seconds'], 'resolution':'1024x1024'})
            print(f'GPU_{i}_PNG_VERIFIED',destination,'render_seconds',entry['seconds'],flush=True)
        timings=json.loads((DIRECT_ROOT/'encoder-timing.json').read_text())
        report={'status':'DUAL_GPU_NOTEBOOK_RENDER_VERIFIED','source_sha':info['sha'],
                'encoder_gpu':0,'encoding_seconds':timings['seconds'],
                'model_startup_seconds':DIRECT_STARTUP_SECONDS,
                'parallel_inference_wall_seconds':DIRECT_INFERENCE_WALL_SECONDS,
                'images':artifacts,'generated_at':datetime.now(timezone.utc).isoformat(),
                'proof':'Each independent worker reached PIPELINE_READY and returned a SHA256-verified 1024x1024 PNG'}
        temp = DIRECT_REPORT.with_suffix('.json.tmp')
        temp.write_text(json.dumps(report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        os.replace(temp,DIRECT_REPORT)
        DIRECT_CACHED_REPORT=report
        DIRECT_SHOULD_RENDER=False
        print('NOTEBOOK_DUAL_GPU_DEMONSTRATION_COMPLETE',DIRECT_REPORT,flush=True)
    except BaseException as error:
        print('NOTEBOOK_GENERATION_ERROR',type(error).__name__,str(error),flush=True)
        for i,p in enumerate(worker_logs):
            if p.exists():
                print('\n=== GPU',i,'WORKER LOG (last 35 lines) ===',flush=True)
                print('\n'.join(p.read_text(errors='replace').splitlines()[-35:]),flush=True)
        raise
    finally:
        for proc in worker_processes:
            if proc.poll() is None:
                os.killpg(proc.pid,signal.SIGTERM)
        for proc in worker_processes:
            try: proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid,signal.SIGKILL)
                proc.wait(timeout=10)
        for handle in worker_handles:
            handle.close()


## 06 · Generated images and measured performance

The images below originate from **this notebook-native workflow**, either freshly generated or reused from a previously verified run. They are never copied from Studio or the public galleries. Each PNG is checked for 1024 × 1024 resolution and SHA256 integrity. The report shows worker GPU assignment, prompt, seed, measured image-generation seconds, the encoder phase, model startup time, and parallel execution wall time.

A dual-GPU success result is reported **only after both workers have loaded their model and each worker has returned a verified image**. GPU import checks or a running API alone do not count.

---

## 06 · Hai ảnh đã sinh và kết quả đo hiệu suất

Hai ảnh bên dưới thuộc **quy trình notebook-native**, được sinh mới hoặc tái sử dụng từ lần chạy trước đã xác minh đầy đủ. Ảnh không lấy từ Studio hay gallery công khai. Mỗi PNG được kiểm tra kích thước 1024 × 1024 và SHA256. Báo cáo hiển thị GPU thực hiện, prompt, seed, thời gian sinh ảnh, thời gian mã hóa prompt, thời gian nạp model và thời gian chạy song song.

Chỉ ghi nhận thành công dual-GPU khi **hai worker đều nạp xong model và mỗi worker trả về một ảnh hợp lệ**. Kiểm tra import thư viện hoặc API đang hoạt động không đủ để chứng minh inference.


In [ ]:
from IPython.display import display, HTML
import base64

result = direct_previous_result()
if result is None:
    raise RuntimeError('NO_NOTEBOOK_IMAGES: run Steps 04 and 05. Studio images are not substituted.')
assert result['status'] == 'DUAL_GPU_NOTEBOOK_RENDER_VERIFIED'
items=sorted(result['images'],key=lambda x:x['gpu'])
assert [x['gpu'] for x in items]==[0,1], 'Both GPUs must have a verified image'

columns=[]
for x in items:
    data=base64.b64encode(Path(x['file']).read_bytes()).decode('ascii')
    import html
    prompt=html.escape(x['prompt'])
    columns.append(f"""<div style="flex:1;min-width:280px;background:#151b29;color:white;border:1px solid #374151;border-radius:14px;padding:15px">
    <h3 style="margin:0 0 10px">GPU {x['gpu']} · Independent GGUF Q6_K</h3>
    <img src="data:image/png;base64,{data}" style="width:100%;max-width:512px;border-radius:8px" />
    <p><b>1024 × 1024 PNG</b> · Seed {x['seed']} · {x['generation_seconds']} seconds</p>
    <p style="font-size:13px;line-height:1.55">{prompt}</p>
    <small>SHA256: {x['sha256'][:24]}…</small></div>""")
display(HTML('<div style="display:flex;gap:16px;flex-wrap:wrap">' + ''.join(columns) + '</div>'))
print('NOTEBOOK_OWN_INFERENCE_PROOF: PASS — 2 independent model processes, 2 verified 1024×1024 images')
print('NOTEBOOK_IMAGE_PROVENANCE:', 'HASH_VERIFIED_REUSE_FROM_PREVIOUS_RUN' if DIRECT_CACHED_REPORT is not None else 'GENERATED_IN_CURRENT_RUN')
print('Source SHA:',result['source_sha'])
print('Text encoding on GPU 0:',result['encoding_seconds'],'seconds')
print('Two model instances startup:',result['model_startup_seconds'],'seconds')
print('Parallel generation wall time:',result['parallel_inference_wall_seconds'],'seconds')
for x in items:
    print(f"GPU {x['gpu']}: PID {x['pid']}, seed {x['seed']}, generation {x['generation_seconds']}s, SHA256 {x['sha256']}")
print('Output directory:',DIRECT_ROOT)


## Optional · Explore Z-Image Studio and public galleries

The notebook GPU demonstration above is complete once both original PNGs have been displayed. The following cells provide a separate authenticated Web Studio and two public HTTPS galleries. They are **optional product experiences**, not prerequisites for the notebook GPU result and not substitutes for the direct benchmark.

---

## Tùy chọn · Trải nghiệm Z-Image Studio và thư viện ảnh

Phần minh chứng notebook hoàn thành sau khi hai ảnh PNG gốc được hiển thị. Các cell tiếp theo cung cấp Web Studio có đăng nhập và hai gallery HTTPS công khai. Đây là **các trải nghiệm bổ sung**, không phải điều kiện bắt buộc hoặc bằng chứng thay thế cho benchmark trực tiếp của notebook.


## Optional · Open Z-Image Studio

Launch the authenticated Studio web application as an optional, browser-based experience. The notebook has already performed its own inference in Steps 04–06.

---

## Tùy chọn · Mở Z-Image Studio

Khởi động Web Studio có đăng nhập để trải nghiệm trên trình duyệt. Notebook đã tự chạy inference và hiển thị ảnh ở các bước 04–06.


In [ ]:
import secrets,json,time,urllib.request,shutil
from pathlib import Path
PROD=PROJECT/"production-system"
RUN=PROD/"b3-studio-demo-live-20261016"
cfgpath=RUN/"local-secrets.json"
if REUSE_LIVE_STUDIO:
    assert RUN.is_dir() and cfgpath.is_file(), "Previously verified Studio runtime is missing"
    print("STUDIO_ALREADY_RUNNING: reusing verified API and gateway", flush=True)
elif not RUN.exists():
    import socket
    occupied=[]
    for port in (18788,18789):
        with socket.socket() as sock:
            sock.settimeout(0.5)
            if sock.connect_ex(("127.0.0.1",port))==0:
                occupied.append(port)
    if occupied:
        raise RuntimeError(
            "STUDIO_PORT_CONFLICT: "+str(occupied)+
            " already in use by an earlier run. Stop the old Studio processes "
            "or restart the Kaggle runtime before starting this checkout."
        )
    subprocess.run([sys.executable,str(PROD/"studio-demo/start_local_demo.py")],check=True,env=os.environ.copy())
else:
    assert cfgpath.exists(),"Incomplete runtime: manual inspection required"
    subprocess.run([sys.executable,str(PROJECT/"deployment"/"resume_local_demo.py")],check=True,env=os.environ.copy())
cfg=json.loads(cfgpath.read_text())
assert all(cfg.get(key) for key in ("api_token","studio_password","service_key"))
assert cfgpath.stat().st_mode & 0o077==0,"Unsafe secret permissions"
for port,path in ((18788,"/health"),(18789,"/auth/session")):
    with urllib.request.urlopen(f"http://127.0.0.1:{port}{path}",timeout=10) as res:
        assert res.status==200
print("STUDIO_GATEWAY_READY: http://127.0.0.1:18789/studio/")
print("Service keys are NOT printed")


## Optional · Enable interactive Studio generation

Start the separate Studio job consumer for interactive prompt submissions. Studio tasks are independent of the two images generated above.

---

## Tùy chọn · Bật tính năng sinh ảnh trong Studio

Khởi động bộ xử lý hàng đợi riêng của Studio để nhận prompt từ giao diện web. Các job Studio độc lập với hai ảnh mà notebook đã sinh phía trên.


In [ ]:
import os, sys, sqlite3, time, subprocess
from pathlib import Path

assert "RUN" in globals() and "PROJECT" in globals(), "Run Steps 1–4 first"
DB=RUN/"api.sqlite3"
LOG=RUN/"persistent-gpu-consumer.log"
PIDFILE=RUN/"persistent-gpu-consumer.pid"
assert DB.is_file(), "STUDIO_DB_MISSING: Step 4 must initialize the database"

def snapshot_queue():
    with sqlite3.connect("file:"+str(DB)+"?mode=ro",uri=True,timeout=5) as con:
        con.row_factory=sqlite3.Row
        states={s:n for s,n in con.execute("SELECT state,count(*) FROM jobs GROUP BY state")}
        sessions=[dict(x) for x in con.execute("SELECT session_id,status FROM b112_sessions WHERE status!='ARCHIVED'")]
        return states,sessions

def consumer_health():
    if not PIDFILE.is_file():return False,"PID_FILE_MISSING"
    try:pid=int(PIDFILE.read_text().strip())
    except (ValueError,OSError):return False,"INVALID_PID"
    proc=Path("/proc")/str(pid)
    if not proc.is_dir():return False,"PROCESS_NOT_RUNNING"
    try:
        stat=(proc/"stat").read_text().split()
        if stat[2] in ("Z","X"):return False,"DEAD_OR_ZOMBIE_PROCESS"
        argv=(proc/"cmdline").read_bytes().replace(b"\x00",b" ").decode(errors="replace")
        if "demo_consumer.py" not in argv or "--enable-gpu" not in argv:
            return False,"PID_BELONGS_TO_UNRELATED_PROCESS"
        env=(proc/"environ").read_bytes().split(b"\x00")
        expected=("ZIMAGE_B1_DB="+str(DB)).encode()
        if expected not in env:return False,"CONSUMER_USES_DIFFERENT_DATABASE"
    except OSError as exc:
        return False,"CANNOT_INSPECT_CONSUMER:"+type(exc).__name__
    if not LOG.is_file() or "STUDIO_CONSUMER_READY" not in LOG.read_text(errors="replace"):
        return False,"CONSUMER_STARTUP_NOT_CONFIRMED"
    return True,"PID="+str(pid)+" DB_MATCH READY_MARKER"

def show_consumer_tail(lines=75):
    print("=== GPU CONSUMER LOG ===")
    if LOG.is_file():print("\n".join(LOG.read_text(errors="replace").splitlines()[-lines:]))
    else:print("CONSUMER_LOG_MISSING")

states,sessions=snapshot_queue()
print("QUEUE_BEFORE_GPU_TEST",states,"UNRESOLVED_SESSIONS",sessions)
assert not sessions,"PREEXISTING_UNRESOLVED_SESSION: inspect this session; no new GPU jobs allowed"
assert not any(states.get(s,0) for s in ("queued","running","uncertain","failed")), (
    "PREEXISTING_UNRESOLVED_JOBS: inspect/cancel or archive them before running fresh demo")

ok,reason=consumer_health()
if not ok:
    if PIDFILE.exists():
        show_consumer_tail()
        raise RuntimeError("EXISTING_CONSUMER_NOT_READY:"+reason+". Do not launch a duplicate consumer")
    print("STARTING_GPU_CONSUMER (not yet claiming inference PASS)")
    log_start=LOG.stat().st_size if LOG.exists() else 0
    p=subprocess.run([sys.executable,str(PROJECT/"deployment/start_persistent_gpu_consumer.py")],
        capture_output=True,text=True,env=os.environ.copy(),timeout=150)
    if p.returncode:
        print(p.stdout[-1200:],p.stderr[-2000:]);show_consumer_tail()
        raise RuntimeError("GPU_CONSUMER_START_FAILED")
    print(p.stdout.strip())
    until=time.monotonic()+50
    while time.monotonic()<until:
        ok,reason=consumer_health()
        if ok:
            # Old READY markers in append-only logs must NOT validate a new process.
            with LOG.open("rb") as fp:
                fp.seek(log_start)
                new_log=fp.read().decode(errors="replace")
            if "STUDIO_CONSUMER_READY" in new_log:break
            ok=False;reason="NEW_PROCESS_READY_MARKER_NOT_YET_OBSERVED"
        if reason in ("PROCESS_NOT_RUNNING","DEAD_OR_ZOMBIE_PROCESS","PID_BELONGS_TO_UNRELATED_PROCESS"):
            break
        time.sleep(1)
    if not ok:
        show_consumer_tail()
        raise RuntimeError("GPU_CONSUMER_STARTUP_GATE_FAILED: "+reason)

print("IMAGE_GENERATION_COORDINATOR_ONLINE",reason)
print("MODEL_WORKERS_START_WITH_LIVE_JOBS: Submit a Studio prompt, then observe the final Live Monitor cell")


## Optional · Browse the 48-image and 122-image galleries

Restore the published visual collections from the official Release archive and run the two read-only gallery webservers.

---

## Tùy chọn · Khám phá gallery 48 ảnh và 122 ảnh

Khôi phục hai bộ sưu tập ảnh công khai từ archive của Release và chạy hai webserver gallery chỉ đọc.


In [ ]:
import hashlib, tarfile, shutil, urllib.request, http.server, socketserver, threading, re
from pathlib import Path
GALLERY_NAME="zimage-gallery-170-watermarked-public-v1.0.0.tar.gz"
EXPECTED_SHA256="e0240d7bc60314f9e8394b5afd922a330216ecd6b596122e381f0bab3e755997"
URL="https://github.com/dangkhoa2016/Z-Image-Turbo-GGUF-Dual-On-Kaggle-T4x2-GPU/releases/download/v1.0.0/"+GALLERY_NAME
ARCHIVE=Path("/kaggle/working")/GALLERY_NAME
GROOT=Path("/kaggle/working/zimage-gallery-showcase")
STAGE=Path("/kaggle/working/.zimage-gallery-170-staging")
ready={}
def file_sha256(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for piece in iter(lambda:f.read(1024*1024),b""): h.update(piece)
    return h.hexdigest()
def restore_approved_galleries():
    if not ARCHIVE.is_file() or file_sha256(ARCHIVE)!=EXPECTED_SHA256:
        urllib.request.urlretrieve(URL,ARCHIVE)
    assert file_sha256(ARCHIVE)==EXPECTED_SHA256,"Gallery archive SHA256 mismatch"
    if not GROOT.exists():
        assert not STAGE.exists(),"Incomplete staging directory"
        STAGE.mkdir()
        try:
            with tarfile.open(ARCHIVE,"r:gz") as tf:
                for member in tf:
                    assert member.isfile() or member.isdir(),"Unsupported TAR entry"
                    dest=(STAGE/member.name).resolve()
                    assert dest.is_relative_to(STAGE.resolve()),"Unsafe TAR path"
                    if member.isdir():dest.mkdir(parents=True,exist_ok=True)
                    else:
                        dest.parent.mkdir(parents=True,exist_ok=True)
                        with tf.extractfile(member) as src,dest.open("wb") as dst:shutil.copyfileobj(src,dst)
            extracted=STAGE/"zimage-gallery-170"
            assert extracted.is_dir() and not GROOT.exists()
            extracted.rename(GROOT)
        finally: shutil.rmtree(STAGE,ignore_errors=True)
    checksums=(GROOT/"SHA256SUMS.txt").read_text().splitlines()
    assert len(checksums)>170,"Incomplete gallery checksum manifest"
    for line in checksums:
        expected,rel=line.split("  ",1)
        path=(GROOT/rel).resolve()
        assert path.is_relative_to(GROOT.resolve()) and path.is_file(),"Missing gallery asset: "+rel
        assert file_sha256(path)==expected,"Gallery checksum mismatch: "+rel
    assert len(list((GROOT/"gallery-48/images").glob("*.png")))==48
    assert len(list((GROOT/"gallery-122/output").rglob("*.png")))==122
    assert len(list((GROOT/"gallery-48/thumbs").glob("*.webp")))==40
    assert (GROOT/"gallery-48/index.html").is_file()
    assert (GROOT/"gallery-122/all-122-gallery/index.html").is_file()
    ready.update({"quality":18881,"benchmark":18882})
    print("GALLERY_170_SHA256_VERIFIED: 48 quality PNG + 122 benchmark PNG + 40 WebP thumbnails")
try:
    restore_approved_galleries()
except Exception as exc:
    ready.clear()
    print("OPTIONAL_GALLERY_170_UNAVAILABLE",type(exc).__name__,str(exc)[:250])
if ready:
    from urllib.parse import urlsplit
    from functools import partial
    gallery_urls={}
    class GalleryHandler(http.server.SimpleHTTPRequestHandler):
        ROOTS={"quality":GROOT/"gallery-48","benchmark":GROOT/"gallery-122"}
        def __init__(self,*args,gallery_kind,**kwargs):
            self.gallery_kind=gallery_kind
            super().__init__(*args,directory=str(self.ROOTS[gallery_kind]),**kwargs)
        def translate_path(self,path):
            route=urlsplit(path).path
            if self.gallery_kind=="benchmark" and not route.startswith("/output/"):
                route="/all-122-gallery"+route
            return super().translate_path(route)
        def do_GET(self):
            route=urlsplit(self.path).path
            if route in ("/","/index.html"):
                rel="index.html" if self.gallery_kind=="quality" else "all-122-gallery/index.html"
                html=(self.ROOTS[self.gallery_kind]/rel).read_text(encoding="utf-8")
                html=html.replace("/gallery-48/index.html",gallery_urls.get("quality","/"))
                html=html.replace("/gallery-122/all-122-gallery/index.html",gallery_urls.get("benchmark","/"))
                # Adapt the bundled gallery navigation to the two live tunnel URLs.
                # Keep archive bytes and release checksums unchanged.
                navigation_js=(
                    "function urlTo(g){\n"
                    " const root=g===\"48\"?"+repr(gallery_urls.get("quality","/"))+":"+repr(gallery_urls.get("benchmark","/"))+";\n"
                    " return root+\"?lang=\"+lang;\n"
                    "}"
                )
                html,replaced=re.subn(r"function urlTo\(g\)\s*\{[\s\S]*?\n\}",lambda m:navigation_js,html,count=1)
                assert replaced==1,"Expected one archived urlTo(g) navigation function"
                assert "function urlTo(g)" in html
                if gallery_urls.get("quality") and gallery_urls.get("benchmark"):
                    assert gallery_urls["quality"] in html and gallery_urls["benchmark"] in html
                body=html.encode("utf-8")
                self.send_response(200)
                self.send_header("Content-Type","text/html; charset=utf-8")
                self.send_header("Content-Length",str(len(body)))
                self.end_headers()
                self.wfile.write(body)
            else:
                return super().do_GET()
    class ReusableServer(socketserver.ThreadingTCPServer):
        allow_reuse_address=True
    gallery_servers={}
    for kind,port in (("quality",18881),("benchmark",18882)):
        server=ReusableServer(("127.0.0.1",port),partial(GalleryHandler,gallery_kind=kind))
        threading.Thread(target=server.serve_forever,daemon=True).start()
        gallery_servers[kind]=server
        assert urllib.request.urlopen("http://127.0.0.1:"+str(port)+"/",timeout=5).status==200
    print("GALLERY_SERVERS_READY quality=18881 benchmark=18882")
else:
    print("GALLERY_SERVERS_SKIPPED")


## Optional · Share the gallery websites

Publish temporary HTTPS links for the quality gallery and benchmark gallery, with working navigation between them.

---

## Tùy chọn · Chia sẻ hai website gallery

Tạo hai đường dẫn HTTPS tạm cho gallery chất lượng và gallery benchmark, kèm liên kết điều hướng giữa hai trang.


In [ ]:
import os, re, stat, subprocess, tempfile, time, urllib.request
from pathlib import Path

CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
cf = Path("/kaggle/working/cloudflared")

def cloudflared_version(binary):
    try:
        proc = subprocess.run(
            [str(binary), "--version"], capture_output=True, text=True,
            check=True, timeout=15
        )
        return proc.stdout.strip() or proc.stderr.strip()
    except (OSError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
        return None

def ensure_cloudflared():
    if cf.exists() and not cf.is_file():
        raise RuntimeError("Cloudflared path exists but is not a regular file: " + str(cf))
    if cf.is_file():
        # Every execution needs this check, not only the first download.
        cf.chmod(cf.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
        version = cloudflared_version(cf)
        if version:
            print("CLOUDFLARED_READY", version, flush=True)
            return cf
        print("CLOUDFLARED_REINSTALL: existing executable failed version check", flush=True)

    fd, temporary = tempfile.mkstemp(prefix=".cloudflared-", dir=str(cf.parent))
    os.close(fd)
    try:
        urllib.request.urlretrieve(CLOUDFLARED_URL, temporary)
        os.chmod(temporary, 0o755)
        version = cloudflared_version(Path(temporary))
        if not version:
            raise RuntimeError(
                "Downloaded cloudflared cannot execute. Check Kaggle runtime "
                "architecture, file permissions and executable filesystem."
            )
        os.replace(temporary, cf)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)
    print("CLOUDFLARED_READY", version, flush=True)
    return cf

cf = ensure_cloudflared()
def quick_tunnel(name,port):
    logfile=Path(f"/kaggle/working/{name}-cloudflared.log")
    logfile.write_text("")  # Discard stale Quick Tunnel URLs from earlier notebook runs.
    proc=subprocess.Popen([str(cf),"tunnel","--no-autoupdate","--url",f"http://127.0.0.1:{port}"],stdout=logfile.open("ab"),stderr=subprocess.STDOUT,start_new_session=True)
    print("TUNNEL_PID",proc.pid)
    for attempt in range(50):
        text=logfile.read_text(errors="replace")
        urls=re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com",text)
        if urls and proc.poll() is None:return urls[-1]
        if proc.poll() is not None:raise RuntimeError(f"{name} tunnel exited: {text[-1200:]}")
        time.sleep(1)
    raise RuntimeError(f"Tunnel URL not found in {logfile}")
ready=globals().get("ready",{})
if ready:
    gallery_urls["quality"]=quick_tunnel("gallery-quality",18881)
    gallery_urls["benchmark"]=quick_tunnel("gallery-benchmark",18882)
    print("QUALITY_GALLERY_HTTPS:",gallery_urls["quality"]+"/")
    print("BENCHMARK_GALLERY_HTTPS:",gallery_urls["benchmark"]+"/")
    for kind,port in (("quality",18881),("benchmark",18882)):
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/",timeout=10) as response:
            served=response.read().decode("utf-8")
        assert gallery_urls["quality"] in served and gallery_urls["benchmark"] in served
        print("GALLERY_CROSS_DOMAIN_NAV_LOCAL_PASS",kind)
    # Public HTTPS check catches stale tunnels and routing errors before claiming success.
    for kind,url in gallery_urls.items():
        last_error=None
        for retry in range(4):
            try:
                with urllib.request.urlopen(url+"/",timeout=12) as res:
                    assert res.status==200
                    served=res.read().decode("utf-8")
                assert gallery_urls["quality"] in served and gallery_urls["benchmark"] in served
                last_error=None
                break
            except Exception as err:
                last_error=err
                time.sleep(2)
        if last_error is not None:
            raise RuntimeError("PUBLIC_GALLERY_HTTPS_CHECK_FAILED: "+kind+" "+str(last_error))
        print("GALLERY_CROSS_DOMAIN_NAV_HTTPS_PASS",kind)
else:
    print("GALLERY_TUNNELS_SKIPPED: gallery archive unavailable")

# Present both public gallery destinations directly in the notebook.
if ready:
    from IPython.display import display, Markdown
    display(Markdown(
        "### Gallery links / Liên kết gallery\n\n"
        "- **Quality · 48 images:** [Open gallery / Mở gallery](" + gallery_urls["quality"] + "/)\n"
        "- **Benchmark · 122 images:** [Open gallery / Mở gallery](" + gallery_urls["benchmark"] + "/)\n\n"
        "Both galleries have independent temporary HTTPS domains."))


## Optional · Open the authenticated Web Studio

Publish a temporary HTTPS address for Studio. The notebook prints a disposable demonstration login. No Studio-generated image is used to substitute for the direct GPU result.

---

## Tùy chọn · Mở Web Studio có đăng nhập

Tạo địa chỉ HTTPS tạm cho Studio. Notebook hiển thị tài khoản demo dùng tạm. Ảnh do Studio sinh không được thay thế bằng chứng inference trực tiếp của notebook.


In [ ]:
assert urllib.request.urlopen("http://127.0.0.1:18789/auth/session",timeout=8).status==200
studio_url=quick_tunnel("studio",18789)
print("======================================")
print("Z-IMAGE STUDIO DEMO")
print("URL:      ",studio_url+"/studio/")
print("Username:  studio")
print("Password: ",cfg["studio_password"])
print("======================================")
print("This is a publicly reachable demo login; do not use it for sensitive workloads.")

from IPython.display import display, Markdown
display(Markdown(
    "### Z-Image Studio / Studio trực tuyến\n\n"
    "[Open Studio / Mở Studio](" + studio_url + "/studio/)\n\n"
    "**Username / Tài khoản:** `studio` · **Password / Mật khẩu:** `"
    + cfg["studio_password"] + "`\n\n"
    "*Temporary demonstration login / Đăng nhập demo tạm thời.*"))


## Export the notebook results

Your newly generated GPU images and provenance report remain in `/kaggle/working/zimage-notebook-dual-gpu-demo/`. Download the two PNG files, `report.json`, `encoder.log`, `worker-0.log`, and `worker-1.log` before ending an ephemeral Kaggle session. Saving the executed notebook also preserves the rendered images in its output.

---

## Xuất kết quả demo

Ảnh sinh bởi notebook và thông tin kiểm chứng được lưu trong `/kaggle/working/zimage-notebook-dual-gpu-demo/`. Hãy tải hai PNG, `report.json`, `encoder.log`, `worker-0.log` và `worker-1.log` trước khi tắt phiên Kaggle. Lưu notebook đã thực thi cũng giúp giữ lại hai ảnh trong output của các cell.


## Optional · Live Studio activity and GPU logs

Once Studio is open, run the next cell to follow image-generation requests directly in the notebook. Every **10 seconds**, it reports the queue, session status, active GPU workers, and newly available inference logs. GPU utilization and VRAM are sampled periodically.

This view is **read-only**: it does not submit or replay jobs and does not replace the two-image notebook inference demonstration above.

**Keep the cell running while using Studio.** You can interrupt **this cell only** when you finish monitoring. The Studio services remain independent; Kaggle session lifetime and GPU quotas still apply.

---

## Tùy chọn · Theo dõi hoạt động Studio và nhật ký GPU trực tiếp

Sau khi mở Studio, chạy cell tiếp theo để quan sát trực tiếp các yêu cầu sinh ảnh trong notebook. **Mỗi 10 giây**, cell hiển thị trạng thái hàng đợi, phiên xử lý, GPU worker đang hoạt động và các dòng log inference mới. Mức sử dụng GPU và VRAM cũng được lấy mẫu định kỳ.

Đây là chế độ **chỉ đọc**: không tạo hoặc chạy lại job và không thay thế phần chứng minh notebook đã tự sinh hai ảnh ở phía trên.

**Giữ cell này ở trạng thái đang chạy khi sử dụng Studio.** Khi không cần theo dõi nữa, bạn có thể ngắt **riêng cell này**. Các dịch vụ Studio vẫn chạy độc lập; thời hạn phiên và giới hạn GPU của Kaggle vẫn được áp dụng.


In [ ]:
# Studio live activity monitor: read-only, no GPU jobs are submitted.
# Keep this cell running while using Studio; interrupt this CELL to stop monitoring.
import os
import re
import sqlite3
import subprocess
import time
from datetime import datetime
from pathlib import Path

STUDIO_POLL_INTERVAL_SECONDS = 10
STUDIO_MONITOR_MAX_MINUTES = None  # None: until you interrupt this cell
STUDIO_GPU_METRIC_EVERY_POLLS = 3

assert 2 <= STUDIO_POLL_INTERVAL_SECONDS <= 120
assert STUDIO_MONITOR_MAX_MINUTES is None or STUDIO_MONITOR_MAX_MINUTES > 0
assert "RUN" in globals(), "Start Studio and its GPU consumer before monitoring."
studio_db = Path(RUN) / "api.sqlite3"
consumer_log = Path(RUN) / "persistent-gpu-consumer.log"
consumer_pid_file = Path(RUN) / "persistent-gpu-consumer.pid"
assert studio_db.is_file(), "Studio database not found. Run the Studio setup cells first."

def _studio_consumer_status():
    try:
        pid = int(consumer_pid_file.read_text().strip())
        proc = Path("/proc") / str(pid)
        stat_fields = (proc / "stat").read_text().rsplit(")", 1)[1].split()
        if not stat_fields or stat_fields[0] in ("Z", "X"):
            return "STOPPED", pid
        args = (proc / "cmdline").read_bytes().replace(b"\0", b" ").decode(errors="replace")
        if "demo_consumer.py" not in args or "--enable-gpu" not in args:
            return "UNVERIFIED_PID", pid
        environment = (proc / "environ").read_bytes().split(b"\0")
        if ("ZIMAGE_B1_DB=" + str(studio_db)).encode() not in environment:
            return "DB_MISMATCH", pid
        return "RUNNING", pid
    except (OSError, ValueError, IndexError):
        return "STOPPED", None

def _studio_db_snapshot():
    con = sqlite3.connect("file:" + str(studio_db) + "?mode=ro", uri=True, timeout=5)
    con.row_factory = sqlite3.Row
    try:
        states = {r["state"]: r["n"] for r in
                  con.execute("SELECT state, COUNT(*) AS n FROM jobs GROUP BY state")}
        recent = [dict(r) for r in con.execute(
            "SELECT id, state, worker_id, created_at, updated_at "
            "FROM jobs ORDER BY created_at DESC LIMIT 6")]
        sessions = [dict(r) for r in con.execute(
            "SELECT session_id, status FROM b112_sessions "
            "ORDER BY created_at DESC LIMIT 3")]
        return states, recent, sessions
    finally:
        con.close()

def _studio_gpu_summary():
    p = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, timeout=8, check=True,
    )
    parts = []
    for line in p.stdout.splitlines():
        fields = [x.strip() for x in line.split(",")]
        if len(fields) == 4:
            index, util, used, total = fields
            parts.append(f"GPU{index}={util}% VRAM={used}/{total}MiB")
    return " | ".join(parts) if parts else "GPU_METRICS_UNAVAILABLE"

def _allowed_studio_log(line):
    # Do not expose credentials, bearer tokens or sensitive request headers.
    lowered = line.lower()
    if any(s in lowered for s in ("password", "api_token", "authorization:",
                                  "bearer ", "service_key", "studio_password")):
        return False
    return any(s in line for s in (
        "STUDIO_CONSUMER_READY", "GPU_", "ZIMAGE_WORKER_STAGE",
        "ENCODE_", "RuntimeError:", "TimeoutError:", "Traceback",
        "ERROR:", "Error:", "GPU_PRIVATE_PUBLICATION"))

def _read_studio_log_increment(path, positions, max_bytes=32768):
    key = str(path)
    try:
        size = path.stat().st_size
        if key not in positions:
            # Begin monitoring only new log lines, not historical output.
            positions[key] = size
            return []
        if size < positions[key]:
            positions[key] = 0
        with path.open("rb") as f:
            f.seek(positions[key])
            data = f.read(max_bytes)
            positions[key] = f.tell()
        lines = data.decode("utf-8", errors="replace").replace("\r", "\n").splitlines()
        return [line.strip()[:700] for line in lines if _allowed_studio_log(line)][-12:]
    except OSError:
        return []

def monitor_studio_live():
    start = time.monotonic()
    positions = {}
    previous_states = {}
    seen_stage = {}
    tick = 0
    deadline = None if STUDIO_MONITOR_MAX_MINUTES is None else (
        start + 60 * STUDIO_MONITOR_MAX_MINUTES)
    print("STUDIO_LIVE_MONITOR_STARTED "
          f"poll_interval={STUDIO_POLL_INTERVAL_SECONDS}s "
          "mode=READ_ONLY (no generation jobs submitted)", flush=True)
    print("Leave this cell running while using Studio. "
          "Interrupt this cell to stop logging without shutting down Studio.",
          flush=True)
    try:
        while deadline is None or time.monotonic() < deadline:
            timestamp = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
            consumer, pid = _studio_consumer_status()
            try:
                states, recent, sessions = _studio_db_snapshot()
            except (sqlite3.Error, OSError) as error:
                print(f"STUDIO_MONITOR_DB_READ_ERROR {type(error).__name__}: {error}",
                      flush=True)
                states, recent, sessions = {}, [], []
            elapsed = round(time.monotonic() - start)
            counts = " ".join(f"{k}={v}" for k, v in sorted(states.items())) or "no_jobs"
            active = " ".join(f"{s['session_id'][-8:]}:{s['status']}"
                              for s in sessions) or "no_sessions"
            print(f"STUDIO_LIVE_POLL at={timestamp} elapsed={elapsed}s "
                  f"consumer={consumer} pid={pid} jobs=[{counts}] "
                  f"sessions=[{active}]", flush=True)

            # Print changes in job state without inventing a completion percentage.
            for job in recent:
                jid = job["id"]
                identity = (job["state"], job["worker_id"])
                if previous_states.get(jid) != identity:
                    print(f"STUDIO_JOB_UPDATE id={jid[:8]} "
                          f"state={job['state']} gpu={job['worker_id']}", flush=True)
                previous_states[jid] = identity

            # Poll the persistent consumer and only the current session worker logs.
            log_paths = [consumer_log]
            for item in sessions:
                session_id = item["session_id"]
                if re.fullmatch(r"b112-[0-9a-f]{32}", session_id):
                    log_paths.extend(sorted(
                        (Path(RUN) / "sessions" / session_id /
                         "gpu-supervisor-logs").glob("worker-*.log")))
            for path in log_paths:
                for line in _read_studio_log_increment(path, positions):
                    print(f"STUDIO_LIVE_LOG source={path.name} {line}", flush=True)

            # Snapshot GPU load every 30s by default, without creating CUDA contexts.
            if tick % STUDIO_GPU_METRIC_EVERY_POLLS == 0:
                try:
                    print("STUDIO_GPU_METRICS " + _studio_gpu_summary(), flush=True)
                except (OSError, subprocess.SubprocessError) as error:
                    print(f"STUDIO_GPU_METRICS_UNAVAILABLE {type(error).__name__}",
                          flush=True)
            tick += 1
            if consumer != "RUNNING":
                print("STUDIO_MONITOR_STOPPED: GPU consumer is not confirmed running. "
                      "Check its log; no automatic restart or job replay was attempted.",
                      flush=True)
                return
            time.sleep(STUDIO_POLL_INTERVAL_SECONDS)
    except KeyboardInterrupt:
        print("STUDIO_LIVE_MONITOR_INTERRUPTED: monitoring stopped; "
              "Studio services were not stopped.", flush=True)
    else:
        print("STUDIO_LIVE_MONITOR_DURATION_COMPLETE: "
              "Studio services were not stopped.", flush=True)

monitor_studio_live()
